# Denoising 3: Conditional Autoencoder (Gu, Kelly & Xiu, 2021) → GDBN

The **conditional autoencoder** (Gu, Kelly & Xiu, "Autoencoder asset pricing models", *Journal of Econometrics*, 2021) is the **nonlinear version of IPCA** (notebook 15), by the authors of the Gu–Kelly–Xiu (2020) paper in my walkthroughs.

```
            BETA NETWORK (nonlinear)                         FACTOR NETWORK (linear)
yesterday's characteristics z (26) per stock          today's returns of 26 "characteristic portfolios"
            |  hidden layer (32, ReLU)                               |  (Z'Z)^-1 Z' r
            v                                                        v
      exposures beta_i (K = 5)        x        today's factors f (K = 5)
                              \                 /
                               shared part of each stock's return  -->  residual = return - shared part  --> GDBN
```

- **Beta network:** turns each stock's characteristics into its factor exposures, through a hidden layer, so it can learn curved relationships (e.g., "only very volatile, small stocks react to factor 2"). IPCA can only learn straight-line ones.
- **Factor network:** the factors are combinations of 26 portfolios that weight stocks by their characteristics (the same building blocks as IPCA).
- **Autoencoder:** today's returns go in (through the portfolios), get squeezed to K = 5 factors (the waist), and come back out as each stock's shared part. Training makes the rebuilt returns as close as possible to the actual ones.

| | Setting |
|---|---|
| Waist | **K = 5** factors (same as PCA and IPCA) |
| Beta network | 26 → 32 (ReLU) → 5; small L1 penalty on weights |
| Training | each year, on the previous 5 years, one day (one cross-section) at a time (up to 60 passes); the last year of the window for early stopping (best out-of-sample R², stop after 5 passes without improvement) |
| Speed | runs on the CPU (faster than the Mac GPU for a model this small) |

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pit_common as pc
import denoise_common as dc

D = pc.load()                        # returns and universe for the GDBN (everything after 2022 blanked)
cube = dc.load_cube()                # the feature table (features_daily.csv.gz) as arrays
GDBN_MASK = dc.gdbn_mask(D, cube)    # the GDBN's 52-stock universe, for R^2 on its own stocks


def frame(arr, rows, cols=None):
    return pd.DataFrame(arr, index=cube.days[rows], columns=cube.perm if cols is None else cols)

## 1. Characteristics and characteristic portfolios

In [ ]:
import torch
import torch.nn as nn
torch.set_num_threads(8)

K, HIDDEN, LR, MAX_EPOCHS, PATIENCE, VAL_DAYS, L1 = 5, 32, 3e-3, 60, 5, 252, 1e-5
Z = dc.rank_chars(cube)
L = Z.shape[2] + 1
nd = len(cube.days)
XP = np.zeros((nd, L))                                       # characteristic-managed portfolio returns
for d in range(1, nd):
    m = cube.AVAIL[d]
    Zd = np.c_[Z[d - 1, m], np.ones(m.sum())]
    XP[d] = np.linalg.solve(Zd.T @ Zd + 1e-6 * np.eye(L), Zd.T @ cube.R[d, m])


def day(d):
    m = cube.AVAIL[d]
    Zd = np.c_[Z[d - 1, m], np.ones(m.sum())].astype(np.float32)
    return m, torch.from_numpy(Zd), torch.from_numpy(XP[d].astype(np.float32)), torch.from_numpy(cube.R[d, m] * 100)


class CondAE(nn.Module):
    def __init__(self, L, K, H):
        super().__init__()
        self.beta = nn.Sequential(nn.Linear(L, H), nn.ReLU(), nn.Linear(H, K))
        self.factor = nn.Linear(L, K, bias=False)

    def forward(self, Zd, xp):
        f = self.factor(xp)
        return self.beta(Zd) @ f, f
print(f"Portfolios ready: {L} per day")

## 2. Train each year and compute residuals

In [ ]:
def val_r2(model, days_):
    num = den = 0.0
    with torch.no_grad():
        for d in days_:
            _, Zd, xp, r = day(d)
            pred, _ = model(Zd, xp)
            num += float(((r - pred) ** 2).sum()); den += float((r ** 2).sum())
    return 1 - num / den


by_year, rows = {}, []
t0 = time.time()
for Y in dc.YEARS:
    fd, ad = dc.fit_days(cube, Y), dc.apply_days(cube, Y)
    train, val = fd[:-VAL_DAYS], fd[-VAL_DAYS:]
    torch.manual_seed(Y)
    rng = np.random.default_rng(Y)
    model = CondAE(L, K, HIDDEN)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    best, state, bad, passes = -np.inf, None, 0, 0
    for epoch in range(MAX_EPOCHS):
        model.train()
        for d in rng.permutation(train):
            _, Zd, xp, r = day(d)
            pred, _ = model(Zd, xp)
            loss = ((r - pred) ** 2).mean() + L1 * sum(p.abs().sum() for p in model.parameters())
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        score = val_r2(model, val)
        passes += 1
        if score > best:
            best, state, bad = score, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(state)
    model.eval()

    resid = np.full((len(ad), len(cube.perm)), np.nan, dtype=np.float32)
    F = np.zeros((len(ad), K))
    with torch.no_grad():
        for n, d in enumerate(ad):
            m, Zd, xp, r = day(d)
            pred, f = model(Zd, xp)
            resid[n, m] = cube.R[d, m] - pred.numpy() / 100
            F[n] = f.numpy() / 100
    by_year[Y] = SimpleNamespace(RET=frame(resid, ad), SENT=None, ATTN=None,
                                 FACT=pd.DataFrame(F, index=cube.days[ad], columns=[f"cae{k + 1}" for k in range(K)]))
    yd = cube.days[ad].year == Y
    actual, fitted = cube.R[ad][yd], (cube.R[ad] - resid)[yd]
    rows.append({"year": Y, "passes": passes, "best validation R2": best,
                 "R2 S&P members (out of sample)": dc.r2(np.where(cube.MEM[ad][yd], actual, np.nan), fitted),
                 "R2 GDBN universe (out of sample)": dc.r2(np.where(GDBN_MASK[ad][yd], actual, np.nan), fitted)})
    print(f"  {Y}: {passes} passes, validation R2 {best:.3f} ({(time.time() - t0) / 60:.1f} min)")
r2_table = pd.DataFrame(rows).set_index("year")
r2_table.round(3)

**Reading it:** R² = share of each day's excess-return moves explained by the 5 factors on days not used for training. Compare with PCA (notebook 14) and IPCA (notebook 15): if the conditional autoencoder's R² is not higher, the nonlinear beta network adds nothing over IPCA.

## GDBN on the denoised data

Four versions of the GDBN, all on the same 52-stock point-in-time universe, walk-forward, λ rule (best rank IC on 2006–2020), and 2021–2022 validation:

| Version | Inputs (yesterday's values) |
|---|---|
| **raw daily** | excess return, sentiment, attention, ETFs (notebook 06) |
| **raw slow** | past-week and past-month averages of the same (notebook 10, the best so far) |
| **CAE daily** | the same, with the **denoised** series in place of the raw one (the conditional-autoencoder residual return), plus the 5 factor returns as market nodes |
| **CAE slow** | past-week and past-month averages of the denoised series, plus the factors' averages |

The target is unchanged: each stock's next-day excess return. The raw versions are saved the first time and loaded in later notebooks.

**Columns:** `arrows/quarter` = network size at the chosen λ; `arrow stability` = how much consecutive quarters' networks overlap (1 = identical). Denser networks overlap more just by chance, so `arrows @ raw λ` and `stability @ raw λ` repeat both at the **same λ as the raw GDBN** of the same kind: the fair, like-for-like comparison. Then rank IC, Newey–West t, gross Sharpe (before costs), and breakeven bps (the cost level that would wipe out the profit) for each period.

In [ ]:
runs = dc.raw_gdbn(D)
runs["CAE daily"] = dc.walk_forward("cae_daily", D, dc.gdbn_features(by_year, "daily"))
runs["CAE slow"] = dc.walk_forward("cae_slow", D, dc.gdbn_features(by_year, "slow"))
table, scores = dc.evaluate(D, runs)
print(table.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, s in scores.items():
    ic = pc.rank_ic(s, pc.forward_excess(D, s, 1))
    axes[0].plot(ic.cumsum(), label=name, ls="--" if name.startswith("raw") else "-")
for name, res in runs.items():
    k, _ = pc.choose_lambda(D, res, 1)
    axes[1].plot([q["r"] for q in res], [(q["W"][:, :, k] != 0).sum() for q in res], label=name,
                 ls="--" if name.startswith("raw") else "-")
axes[0].axvline(pd.Timestamp("2021-01-01"), color="gray", lw=1); axes[0].set_title("Cumulative rank IC (rising = skill; line = validation starts)")
axes[1].set_title("Arrows in each quarter's network (at each version's chosen λ)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading it:**
- **Did denoising help prediction?** Compare the CAE rows with the raw rows of the same kind (daily vs daily, slow vs slow): rank IC, gross Sharpe, breakeven, and especially **2021–2022**.
- **Did it clean up the connections?** Compare **stability @ raw λ** (same λ, so the same price per arrow): higher means the network keeps finding the same links from quarter to quarter, which is what real (not noise-driven) connections should do. The plain `arrow stability` column is inflated for versions that chose a smaller λ (bigger networks).
- Notebook 18 puts all four denoisers side by side.

In [ ]:
dc.save("cae", table, r2_table)